To run this notebook, you will need a `.env` file at the root of the project. It should contain the following keys:
```
LLM_SERVICE=OpenAI
OPENAI_API_KEY=...
OPENAI_ENDPOINT=...
OPENAI_DEPLOYMENT_NAME=...
OPENAI_API_VERSION=...
```

The first step is to load the libraries we'll be using. I am importing `pandas` for basic data manipulation, and a package called [discovery_utils](https://github.com/nestauk/discovery_utils) that Karlis made. It contains various functions that we have used and reused across Discovery projects, including functions for extracting structured information using LLMs. You can read more about how the `llm` module works [here](https://github.com/nestauk/discovery_utils/wiki/Checking-data-with-LLM).

In [ ]:
import pandas as pd
from discovery_utils.utils.llm import batch_check

from discovery_heat_pump_futures import PROJECT_DIR

Karlis has created two datasets for this project: one containing research abstracts from [OpenAlex](https://openalex.org/), and one from [Google Patents](https://patents.google.com/).

In [ ]:
openalex_df = pd.read_csv(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
)
patents_df = pd.read_json(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json",
    lines=True,
)

The patents dataset looks like this:

In [ ]:
patents_df.head()

Below, we specify a helper function that can also be used on the OpenAlex data to concatenate the titles and abstracts of individual patents/abstracts. This new `"title_abstract"` field will be the input to `LLMProcessor`.

In [ ]:
def format_title_abstract(
    df: pd.DataFrame, title: str = "title", abstract: str = "abstract"
) -> pd.DataFrame:
    """Format the title and abstract for LLM input.

    Args:
        df (pd.DataFrame): DataFrame containing the title and abstract columns.
        title (str): Name of the title column.
        abstract (str): Name of the abstract column.

    Returns:
        pd.DataFrame: DataFrame with a new column 'title_abstract'
            containing formatted text.
    """
    df["title_abstract"] = (
        "TITLE: "
        + df[title].str.lower().fillna("")
        + " ABSTRACT: "
        + df[abstract].str.lower().fillna("")
    )
    return df


patents_df = format_title_abstract(patents_df, title="title", abstract="abstract")
patents_df.head()

In [ ]:
# to show what the formatted column looks like
patents_df["title_abstract"].values[0]

We need to define a system message to tell the LLM what it should do, and define exactly what outputs we want to get back, and what format they should be in.

In [ ]:
system_message = """
Determine whether this text presents an improvement to heat pump components or systems.
"""

fields = [
    {
        "name": "is_relevant",
        "type": "str",
        "description": "A one-word answer: 'yes' or 'no'.",
    },
    {
        "name": "summary",
        "type": "str",
        "description": "Summarise the patent in plain English.",
    },
    {
        "name": "components",
        "type": "List[str]",
        "description": """
        In terms of heat pump components,
         does this application mention refrigerants,
         the mechanical compressor, or heat exchange?
         Return all that apply.""",
    },
]

We will create a small sample of data and convert it into the correct format for passing to the LLM:

In [ ]:
patents_sample = patents_df.sample(5, random_state=42)

test_data = patents_sample[["url", "title_abstract"]]

test_dict = test_data.set_index("url")["title_abstract"].to_dict()
test_dict

Now that we have test data, a system message and some defined output fields, we are ready to run `LLMProcessor`.

In [ ]:
outpath = PROJECT_DIR / "outputs/llm_check_output.jsonl"

processor = batch_check.LLMProcessor(
    model_name="gpt-4o-mini",
    temperature=0,
    output_path=str(outpath),
    system_message=system_message,
    session_name="test",
    output_fields=fields,
)

task = processor.run(test_dict, batch_size=1, sleep_time=0.5)
await task

We can now read the output back in to see what we got!

In [ ]:
test_output = pd.read_json(outpath, lines=True)
test_output.head()

In [ ]:


# Download OpenAlex dataset
openalex_df = pd.read_csv(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
)

# Download Patents dataset
patents_df = pd.read_json(
    "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json",
    lines=True,
)

# Optional: Save locally for offline reuse
openalex_df.to_csv("heat_pumps_openalex.csv", index=False)
patents_df.to_json("heat_pumps_patents.json", orient="records", lines=True)

print("Datasets downloaded and saved locally.")


In [ ]:
# Save a smaller sample of the data
openalex_df.head(500).to_csv("sample_openalex.csv", index=False)
patents_df.head(500).to_json("sample_patents.json", orient="records", lines=True)


In [ ]:
import openai
from openai import OpenAI
client = OpenAI()

In [ ]:
response = client.responses.create(
    model="gpt-4.1",
    instructions="Talk like a pirate.",
    input="Are semicolons optional in JavaScript?",
)

print(response.output_text)

In [ ]:
import pandas as pd
import json
from time import sleep
from dotenv import load_dotenv
import os
from openai import OpenAI

# -------------------------------
# Load environment and OpenAI client
# -------------------------------
load_dotenv(dotenv_path="/home/pascualdiego/projects/DiscoveryHP/.env")  # Adjust path if needed
client = OpenAI()

# -------------------------------
# Configuration
# -------------------------------
CATEGORY = "1.1.1 Compressors"
MAX_RECORDS = 50  # limit for testing
OUTPUT_FILE = "classified_compressor_patents.csv"

# -------------------------------
# Load and filter patent data
# -------------------------------
with open("heat_pumps_patents.json", 'r') as f:
    data = [json.loads(line) for line in f]

df = pd.DataFrame(data)

# Filter by compressor-related keywords
keywords = ["compressor", "rotary", "piston", "scroll", "isentropic", "motor", "pump"]
mask = df['abstract'].str.contains('|'.join(keywords), case=False, na=False)
filtered_df = df[mask].head(MAX_RECORDS).copy()

# -------------------------------
# Prompt generation
# -------------------------------
def make_prompt(title, abstract):
    return f"""
You are an expert in sustainable heating technologies.

Please analyze the following patent text and answer these questions:
1. What is the core technical innovation, in 1–2 sentences?
2. Does it describe an improvement in compressor technology? (Yes/No)
3. Assign a category from the following:
   - Component Innovation
   - System Design
   - Non-Traditional Technology
   - Not Relevant

Patent text:
TITLE: {title}
ABSTRACT: {abstract}
"""

# -------------------------------
# OpenAI classification call
# -------------------------------
def classify_patent(row):
    try:
        prompt = make_prompt(row['title'], row['abstract'])

        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )

        reply = response.choices[0].message.content
        return reply

    except Exception as e:
        print(f"Error for {row['publication_number']}: {e}")
        return f"ERROR: {e}"

# -------------------------------
# Run classification loop
# -------------------------------
results = []
for _, row in filtered_df.iterrows():
    print(f"Processing: {row['publication_number']}")
    response = classify_patent(row)
    results.append({
        "publication_number": row["publication_number"],
        "title": row["title"],
        "abstract": row["abstract"],
        "response": response
    })
    sleep(1.5)  # Respect OpenAI rate limits

# -------------------------------
# Save results
# -------------------------------
result_df = pd.DataFrame(results)
result_df.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Saved {len(result_df)} results to: {OUTPUT_FILE}")
